# PoC — Clasificación de intención, extracción de entidades y enrutamiento FAQ

Este notebook implementa la **prueba de concepto (PoC)** del asistente de FAQs de cocina definida en la Entrega 2. El objetivo es demostrar que el pipeline `[Idioma → Intención → Entidades]` funciona correctamente antes de conectarlo con una base de datos real de FAQs.

**Estructura del notebook:**

| Sección | Tarea | Contenido |
|---------|-------|-----------|
| 1 | T2 — Clasificación de intención | Línea base, TF-IDF + LinearSVC, métricas por idioma, matrices de confusión |
| 2 | Análisis de errores | Fallos y conexión con el análisis lingüístico de la Entrega 1 |
| 3 | T3 — Extracción de entidades | Vocabulario controlado (diccionarios culinarios) |
| 4 | Pipeline completo | Función que devuelve `[Idioma, Intención, Entidades]` para cualquier consulta |
| 5 | T4 — Propuesta de enrutamiento | Diseño teórico comparativo: Whoosh (léxico) vs. embeddings (semántico) |

**Requisito previo**: ejecutar `Nuevo_Dataset.ipynb` para generar los archivos `clean_data_*.csv`.

In [ ]:
!pip install -q lingua-language-detector spacy pyspellchecker unidecode contractions whoosh sentence-transformers
!python -m spacy download es_core_news_sm -q
!python -m spacy download en_core_web_sm -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Colab Notebooks/PLN

import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

SEED = 42

# Cargar datos preprocesados
data_train_en = pd.read_csv('clean_data_train_en.csv')
data_test_en  = pd.read_csv('clean_data_test_en.csv')
data_train_es = pd.read_csv('clean_data_train_es.csv')
data_test_es  = pd.read_csv('clean_data_test_es.csv')

INTENT_MAPPING = {
    'meal_suggestion': 1, 'recipe': 2, 'ingredients_list': 3,
    'ingredient_substitution': 4, 'nutrition_info': 5,
    'calories': 6, 'cook_time': 7, 'food_last': 8,
}
REV_INTENT = {v: k for k, v in INTENT_MAPPING.items()}

print(f'Train EN: {len(data_train_en)} | Test EN: {len(data_test_en)}')
print(f'Train ES: {len(data_train_es)} | Test ES: {len(data_test_es)}')

## 1. Clasificación de intención (T2)

### 1.1 Línea base: clasificador trivial

Con **8 clases perfectamente balanceadas**, un clasificador que asigna clases de forma aleatoria y uniforme tiene un rendimiento esperado del **12,5 %** en *accuracy*. Este es el umbral mínimo que debe superar el modelo para justificar su utilidad según los criterios de éxito de la Entrega 2.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

dummy = DummyClassifier(strategy='uniform', random_state=SEED)

# Inglés
dummy.fit(data_train_en['clean_text'], data_train_en['intent'])
y_dummy_en = dummy.predict(data_test_en['clean_text'])
acc_dummy_en = accuracy_score(data_test_en['intent'], y_dummy_en)

# Español
dummy.fit(data_train_es['clean_text'], data_train_es['intent'])
y_dummy_es = dummy.predict(data_test_es['clean_text'])
acc_dummy_es = accuracy_score(data_test_es['intent'], y_dummy_es)

print(f'Accuracy línea base (EN): {acc_dummy_en:.1%}')
print(f'Accuracy línea base (ES): {acc_dummy_es:.1%}')
print(f'Accuracy teórica esperada (1/8 clases): 12.5%')

### 1.2 Modelo TF-IDF + LinearSVC

Se entrena un modelo independiente por idioma. Esta arquitectura separada es coherente con la hipótesis de diseño de la Entrega 1 (T1 detecta el idioma y enruta a un módulo específico). La representación TF-IDF captura la importancia relativa de los términos discriminativos, y el LinearSVC proporciona un clasificador de margen máximo robusto con pocas muestras.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, f1_score

def entrenar_evaluar(train_df, test_df, lang_label):
    vectorizer = TfidfVectorizer()
    model = LinearSVC(C=1, max_iter=1000, random_state=SEED)

    X_train = vectorizer.fit_transform(train_df['clean_text'])
    y_train = train_df['intent']
    model.fit(X_train, y_train)

    X_test = vectorizer.transform(test_df['clean_text'])
    y_test = test_df['intent']
    y_pred = model.predict(X_test)

    clases = sorted(y_test.unique())
    nombres_clases = [REV_INTENT[c] for c in clases]

    print(f'\n=== Resultados [{lang_label}] ===')
    print(classification_report(y_test, y_pred,
                                 labels=clases, target_names=nombres_clases))
    return vectorizer, model, y_test, y_pred

vectorizer_en, model_en, y_test_en, y_pred_en = entrenar_evaluar(
    data_train_en, data_test_en, 'EN')
vectorizer_es, model_es, y_test_es, y_pred_es = entrenar_evaluar(
    data_train_es, data_test_es, 'ES')

### 1.3 Comparación de métricas por idioma

In [ ]:
acc_en = accuracy_score(y_test_en, y_pred_en)
acc_es = accuracy_score(y_test_es, y_pred_es)
f1_en  = f1_score(y_test_en, y_pred_en, average='macro')
f1_es  = f1_score(y_test_es, y_pred_es, average='macro')

df_metricas = pd.DataFrame([
    {'Modelo': 'Clasificador trivial (baseline)',
     'Accuracy EN': f'{acc_dummy_en:.1%}', 'F1 macro EN': '—',
     'Accuracy ES': f'{acc_dummy_es:.1%}', 'F1 macro ES': '—'},
    {'Modelo': 'TF-IDF + LinearSVC',
     'Accuracy EN': f'{acc_en:.1%}', 'F1 macro EN': f'{f1_en:.3f}',
     'Accuracy ES': f'{acc_es:.1%}', 'F1 macro ES': f'{f1_es:.3f}'},
])
print(df_metricas.to_string(index=False))

# Gráfico de barras comparativo
fig, ax = plt.subplots(figsize=(10, 4))
categorias = ['Baseline EN', 'TF-IDF+SVC EN', 'Baseline ES', 'TF-IDF+SVC ES']
valores    = [acc_dummy_en, acc_en, acc_dummy_es, acc_es]
colores    = ['#d4e6f1', '#2980b9', '#d5f5e3', '#27ae60']
bars = ax.bar(categorias, valores, color=colores, edgecolor='white', width=0.6)
for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.axhline(0.125, color='red', linestyle='--', linewidth=1.2, label='Baseline teórico (12.5%)')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Accuracy')
ax.set_title('Comparativa de accuracy por modelo e idioma')
ax.legend()
plt.tight_layout()
plt.savefig('fig_metricas_comparativa.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.4 Matrices de confusión

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
intent_labels = [REV_INTENT[i] for i in sorted(y_test_en.unique())]

for ax, (y_true, y_pred, titulo) in zip(axes, [
    (y_test_en, y_pred_en, f'Inglés — Accuracy: {acc_en:.1%}'),
    (y_test_es, y_pred_es, f'Español — Accuracy: {acc_es:.1%}'),
]):
    cm = confusion_matrix(y_true, y_pred, labels=sorted(y_true.unique()))
    sns.heatmap(
        cm, annot=True, fmt='d', ax=ax, cmap='Blues',
        xticklabels=intent_labels, yticklabels=intent_labels
    )
    ax.set_title(titulo, fontsize=13)
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')
    plt.setp(ax.get_xticklabels(), rotation=40, ha='right', fontsize=9)
    plt.setp(ax.get_yticklabels(), fontsize=9)

plt.suptitle('Matrices de confusión — TF-IDF + LinearSVC', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('fig_matrices_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Análisis de errores

Esta sección conecta los fallos de clasificación con las dificultades lingüísticas identificadas en la **Entrega 1**, verificando empíricamente qué fenómenos predichos resultan problemáticos para el modelo.

In [ ]:
def analizar_errores(test_df, y_pred, lang_label, top_n_pares=5, top_n_ejemplos=2):
    df_err = test_df.copy().reset_index(drop=True)
    df_err['y_pred']    = y_pred
    df_err['correcto']  = df_err['intent'] == df_err['y_pred']
    df_err['pred_name'] = df_err['y_pred'].map(REV_INTENT)

    errores = df_err[~df_err['correcto']].copy()
    total   = len(test_df)

    print(f'\n{'='*70}')
    print(f'ANÁLISIS DE ERRORES [{lang_label}]: {len(errores)} errores de {total} ({len(errores)/total:.1%})')
    print('='*70)

    # Pares de confusión más frecuentes
    pares = (errores
             .groupby(['intent_name', 'pred_name'])
             .size().reset_index(name='count')
             .sort_values('count', ascending=False)
             .head(top_n_pares))

    print(f'\nPares de confusión más frecuentes:')
    print(pares.to_string(index=False))

    print(f'\nEjemplos representativos por par:')
    for _, par in pares.head(3).iterrows():
        mask = ((errores['intent_name'] == par['intent_name']) &
                (errores['pred_name']   == par['pred_name']))
        ejemplos = errores[mask].head(top_n_ejemplos)
        print(f"\n  Real: '{par['intent_name']}' → Predicho: '{par['pred_name']}' (n={par['count']})")
        for _, ej in ejemplos.iterrows():
            print(f"    Texto:   {ej['text']}")
            print(f"    Limpio:  {ej['clean_text']}")

    return errores

errores_en = analizar_errores(data_test_en, y_pred_en, 'EN')
errores_es = analizar_errores(data_test_es, y_pred_es, 'ES')

### 2.1 Conexión con el análisis lingüístico (Entrega 1)

Los errores observados ilustran directamente las dificultades identificadas en la caracterización lingüística:

| Fenómeno (Entrega 1) | Ejemplo de fallo observado | Par de confusión habitual |
|---------------------|---------------------------|---------------------------|
| **Ambigüedad semántica** | *"is it healthy to eat pizza"* (nutrición, no calorías) | `nutrition_info` ↔ `calories` |
| **Petición implícita / pragmática** | *"how healthy is X"* puede pedir info nutricional o calorías | `nutrition_info` ↔ `calories` |
| **Solapamiento léxico entre clases** | *recipe* y *ingredients_list* comparten tokens como `make`, `how` | `recipe` ↔ `ingredients_list` |
| **Traducción literal al español** | Estructuras calco del inglés distorsionan la lematización | Mayor tasa de error en ES |
| **Brevedad / elipsis** | Consultas muy cortas ofrecen pocos tokens discriminativos | Cualquier par con poca señal |

El mayor número de confusiones entre `nutrition_info` y `calories` es coherente con el diagnóstico de **ambigüedad pragmática**: tanto *"how healthy is X"* como *"how many calories in X"* buscan información sobre salud/nutrición, pero el modelo debe inferir el nivel de detalle que espera el usuario (general vs. cuantitativo).

La confusión `recipe` ↔ `ingredients_list` refleja el solapamiento léxico señalado en el análisis: ambas intenciones incluyen verbos como `make` o `prepare` y sustantivos de alimentos, lo que hace difícil distinguirlas sin capturar que `ingredients_list` siempre contiene una consulta por *qué lleva* un plato, mientras que `recipe` pregunta por *cómo hacerlo*. Un modelo TF-IDF sin información sintáctica pierde esta distinción.

## 3. Extracción de entidades (T3)

Como se expuso en la charla del 14 de abril, **la intención es necesaria pero no suficiente**. Saber que el usuario busca una receta es inútil si no se sabe *sobre qué alimento o técnica* pregunta.

T3 contextualiza la intención detectada mediante un **vocabulario controlado**: diccionarios culinarios organizados por categoría que identifican los conceptos clave de la consulta mediante coincidencia de tokens normalizados.

In [ ]:
# Diccionarios culinarios (vocabulario controlado)
ENTIDADES = {
    'ingredientes': {
        'es': {
            # Proteínas
            'pollo', 'ternera', 'cerdo', 'cordero', 'salmon', 'atun', 'gambas',
            'jamon', 'huevo', 'leche', 'queso', 'mantequilla', 'nata', 'yogur',
            # Verduras
            'tomate', 'cebolla', 'ajo', 'patata', 'zanahoria', 'pimiento', 'calabacin',
            'espinaca', 'lechuga', 'brocol', 'coliflor', 'berenjena', 'champiñon',
            # Cereales y legumbres
            'arroz', 'pasta', 'harina', 'pan', 'lentejas', 'garbanzos', 'judias',
            # Condimentos
            'sal', 'pimienta', 'aceite', 'vinagre', 'azucar', 'miel', 'limon',
            'oregano', 'tomillo', 'pimenton', 'comino', 'canela', 'perejil',
        },
        'en': {
            'chicken', 'beef', 'pork', 'lamb', 'salmon', 'tuna', 'shrimp', 'ham',
            'egg', 'milk', 'cheese', 'butter', 'cream', 'yogurt',
            'tomato', 'onion', 'garlic', 'potato', 'carrot', 'pepper', 'zucchini',
            'spinach', 'lettuce', 'broccoli', 'cauliflower', 'eggplant', 'mushroom',
            'rice', 'pasta', 'flour', 'bread', 'lentil', 'chickpea', 'bean',
            'salt', 'oil', 'vinegar', 'sugar', 'honey', 'lemon',
            'oregano', 'thyme', 'paprika', 'cumin', 'cinnamon', 'parsley',
        },
    },
    'tecnicas': {
        'es': {
            'freir', 'hervir', 'hornear', 'asar', 'cocer', 'saltear', 'mezclar',
            'batir', 'amasar', 'marinar', 'gratinar', 'escaldar', 'pochar',
            'caramelizar', 'cocinar', 'preparar', 'hacer', 'cortar', 'picar',
            'triturar', 'moler', 'fundir', 'reducir', 'glasear', 'macerar',
        },
        'en': {
            'fry', 'boil', 'bake', 'roast', 'cook', 'saute', 'mix',
            'beat', 'knead', 'marinate', 'grill', 'blanch', 'poach',
            'caramelize', 'prepare', 'make', 'cut', 'chop',
            'blend', 'grind', 'melt', 'reduce', 'glaze', 'simmer', 'steam',
        },
    },
    'utensilios': {
        'es': {
            'sartan', 'olla', 'cazuela', 'horno', 'airfryer', 'thermomix',
            'batidora', 'licuadora', 'molde', 'colador', 'rallador',
            'cuchillo', 'wok', 'plancha', 'microondas', 'freidora', 'bowl', 'tupper',
        },
        'en': {
            'pan', 'pot', 'oven', 'airfryer', 'thermomix', 'blender',
            'mixer', 'mold', 'colander', 'grater', 'knife',
            'wok', 'grill', 'microwave', 'fryer', 'bowl', 'skillet', 'steamer',
        },
    },
}

def extraer_entidades(texto_limpio: str, lang: str) -> dict:
    """Extrae entidades culinarias por coincidencia de tokens normalizados."""
    tokens = set(texto_limpio.lower().split())
    return {
        cat: sorted(tokens & dic.get(lang, set()))
        for cat, dic in ENTIDADES.items()
        if tokens & dic.get(lang, set())
    }

print('Diccionarios culinarios cargados.')
for cat, dic in ENTIDADES.items():
    print(f'  {cat}: {len(dic["es"])} términos ES | {len(dic["en"])} términos EN')

In [ ]:
# Demostración de extracción de entidades
ejemplos_entidades = [
    ('como freir pollo en sartan con ajo', 'es'),
    ('quiero hornear pasta con queso y tomate en molde', 'es'),
    ('how long to bake salmon in the oven', 'en'),
    ('can i substitute butter with olive oil in a cake recipe', 'en'),
    ('cuanto tiempo tarda en hervir el arroz en olla', 'es'),
    ('give me ingredients for garlic chicken', 'en'),
]

print('EXTRACCIÓN DE ENTIDADES (T3)\n' + '='*60)
for texto, lang in ejemplos_entidades:
    entidades = extraer_entidades(texto, lang)
    print(f'\n[{lang.upper()}] {texto}')
    if entidades:
        for cat, vals in entidades.items():
            print(f'  {cat:15s}: {vals}')
    else:
        print('  (sin entidades detectadas)')

### 3.1 Evaluación empírica de la cobertura de entidades

In [ ]:
# Cobertura: qué porcentaje de ejemplos del test tiene al menos una entidad extraída
for lang_label, test_df, lang_code in [
    ('EN', data_test_en, 'en'),
    ('ES', data_test_es, 'es'),
]:
    test_df['entidades'] = test_df['clean_text'].apply(
        lambda x: extraer_entidades(str(x), lang_code))
    test_df['n_entidades'] = test_df['entidades'].apply(
        lambda d: sum(len(v) for v in d.values()))
    cobertura = (test_df['n_entidades'] > 0).mean()

    print(f'[{lang_label}] Cobertura de entidades en test: {cobertura:.1%}')
    print(f'  Media de entidades por consulta: {test_df["n_entidades"].mean():.2f}')
    print(f'  Consultas sin entidades: {(test_df["n_entidades"] == 0).sum()}')
    print()

# Cobertura por intención en inglés
print('Cobertura por intención (EN):')
cob_intencion = data_test_en.groupby('intent_name')['n_entidades'].agg(
    media='mean', con_entidades=lambda x: (x > 0).sum(), total='count')
cob_intencion['%_cobertura'] = (cob_intencion['con_entidades'] / cob_intencion['total'] * 100).round(1)
print(cob_intencion.sort_values('media', ascending=False).to_string())

## 4. Pipeline completo: `[Idioma, Intención, Entidades]`

Esta sección integra T1 (detección de idioma), T2 (clasificación de intención) y T3 (extracción de entidades) en una función unificada de predicción que, dado el texto libre de un usuario, devuelve la tupla estructurada `[Idioma, Intención, Entidades]`.

```
Texto usuario
     │
     ▼
  T1: Lingua ──────────────── idioma ('es' / 'en')
     │
     ▼
  Preprocesamiento (spaCy, lematización, normalización)
     │
     ├──▶ T2: TF-IDF + LinearSVC ──── intención (1–8)
     │
     └──▶ T3: Diccionario culinario ─ entidades {ingredientes, técnicas, utensilios}
     │
     ▼
  [Idioma, Intención, Entidades]
```

In [ ]:
# Reproducción del pipeline de preprocesamiento (del Notebook 1)
import unidecode
import contractions
import spacy
from spellchecker import SpellChecker
from lingua import Language, LanguageDetectorBuilder

nlp_es = spacy.load('es_core_news_sm')
nlp_en = spacy.load('en_core_web_sm')
detector = LanguageDetectorBuilder.from_languages(Language.SPANISH, Language.ENGLISH).build()

spell_es = SpellChecker(language='es')
spell_en = SpellChecker(language='en')
spell_es.word_frequency.load_words(
    ['aove', 'thermomix', 'airfryer', 'umami', 'keto', 'dente',
     'pizza', 'sushi', 'wok', 'bowl', 'tupper', 'smoothie'])
spell_en.word_frequency.load_words(
    ['bbq', 'thermomix', 'airfryer', 'umami', 'keto', 'ramen'])

KEEP_ES = {'no', 'sin', 'excepto', 'nunca', 'cuando', 'como', 'cuanto',
           'antes', 'despues', 'hacer', 'estar', 'ser', 'buen', 'mal',
           'gluten', 'lactosa', 'proteina', 'vegano', 'vegana', 'keto', 'umami'}
KEEP_EN = {'no', 'not', 'without', 'never', 'free', 'when', 'how',
           'before', 'after', 'make', 'good', 'bad',
           'gluten', 'lactose', 'vegan', 'keto', 'umami', 'bbq'}

STOPWORDS_ES = {unidecode.unidecode(w).lower() for w in nlp_es.Defaults.stop_words} - KEEP_ES
STOPWORDS_EN = {unidecode.unidecode(w).lower() for w in nlp_en.Defaults.stop_words} - KEEP_EN

SLANG_ES = {'aser': 'hacer', 'kiero': 'quiero', 'q': 'que', 'xq': 'porque',
            'komo': 'como', 'weno': 'bueno', 'k': 'que', 'x': 'por'}

EXC_ES = {'gluten': 'gluten', 'proteina': 'proteina', 'lactosa': 'lactosa',
           'vegano': 'vegano', 'vegana': 'vegano', 'bizcocho': 'bizcocho',
           'aove': 'aove', 'pizza': 'pizza'}
EXC_EN = {'gluten': 'gluten', 'ramen': 'ramen', 'vegan': 'vegan'}

SYN_ES = {'papa': 'patata', 'papas': 'patata', 'durazno': 'melocoton',
           'palta': 'aguacate', 'frutilla': 'fresa', 'choclo': 'maiz',
           'elote': 'maiz', 'anana': 'pina', 'jugo': 'zumo',
           'banana': 'platano', 'jitomate': 'tomate', 'batata': 'boniato',
           'camaron': 'gamba', 'mani': 'cacahuete', 'aji': 'chile'}
SYN_EN = {'aubergine': 'eggplant', 'courgette': 'zucchini',
           'coriander': 'cilantro', 'rocket': 'arugula',
           'biscuit': 'cookie', 'yoghurt': 'yogurt',
           'beetroot': 'beet', 'prawn': 'shrimp'}

def apply_synonyms(text, lang):
    synonyms = SYN_ES if lang == 'es' else SYN_EN
    return ' '.join(synonyms.get(w, w) for w in text.split())

def procesar_texto(text, lang, is_predict=False):
    if not isinstance(text, str) or not text.strip():
        return ''
    if lang == 'en':
        text = contractions.fix(text)
    if lang == 'es':
        for s, c in SLANG_ES.items():
            text = re.sub(rf'\b{s}\b', c, text, flags=re.IGNORECASE)
    nlp      = nlp_es if lang == 'es' else nlp_en
    sw       = STOPWORDS_ES if lang == 'es' else STOPWORDS_EN
    keep_set = KEEP_ES if lang == 'es' else KEEP_EN
    spell    = spell_es if lang == 'es' else spell_en
    exc      = EXC_ES if lang == 'es' else EXC_EN
    doc = nlp(text)
    tokens = []
    for tok in doc:
        if tok.is_space or tok.is_punct:
            continue
        sin_tildes = unidecode.unidecode(tok.text).lower()
        lemma = exc.get(sin_tildes, tok.lemma_.lower())
        if is_predict and spell.unknown([lemma]):
            corr = spell.correction(lemma)
            lemma = corr if corr else lemma
        norm = unidecode.unidecode(lemma)
        if norm in keep_set:
            tokens.append(norm)
        elif norm not in sw and len(norm) > 1:
            tokens.append(norm)
    return apply_synonyms(' '.join(tokens), lang)

print('Pipeline de preprocesamiento listo.')

In [ ]:
def predecir_consulta(texto_usuario: str) -> dict:
    """Devuelve la tupla [Idioma, Intención, Entidades] para cualquier consulta."""
    if not isinstance(texto_usuario, str) or not texto_usuario.strip():
        return {'error': 'Texto vacío'}

    # T1: Detección de idioma
    idioma_lingua = detector.detect_language_of(texto_usuario)
    lang = 'es' if idioma_lingua == Language.SPANISH else 'en'

    # Preprocesamiento
    texto_limpio = procesar_texto(texto_usuario, lang=lang, is_predict=True)
    if not texto_limpio:
        return {'error': 'Texto irreconocible tras limpieza', 'idioma': lang}

    # T2: Clasificación de intención
    vectorizer = vectorizer_en if lang == 'en' else vectorizer_es
    model      = model_en      if lang == 'en' else model_es
    vector     = vectorizer.transform([texto_limpio])
    intent_num = int(model.predict(vector)[0])
    intent_name = REV_INTENT.get(intent_num, 'unknown')

    # T3: Extracción de entidades
    entidades = extraer_entidades(texto_limpio, lang)

    return {
        'texto_original': texto_usuario,
        'texto_limpio':   texto_limpio,
        'idioma':         lang,
        'intencion':      f'{intent_num} ({intent_name})',
        'entidades':      entidades,
    }

print('Función predecir_consulta() definida.')

### 4.1 Ejemplos guiados (una consulta por intención)

In [ ]:
consultas_guiadas = [
    ('cuanto tiempo tarda en hacerse el pollo al horno',       'cook_time'),
    ('cuando se me pone malo el queso fresco',                  'food_last'),
    ('que ingredientes necesito para hacer pizza casera',       'ingredients_list'),
    ('puedo sustituir la nata por leche en esta receta',        'ingredient_substitution'),
    ('cuantas calorias tiene un trozo de tarta de chocolate',   'calories'),
    ('me gustaria hacer comida italiana para cenar que me recomiendas', 'meal_suggestion'),
    ('como hacer pasta al dente con tomate y albahaca',         'recipe'),
    ('es sano comer espinacas todos los dias',                  'nutrition_info'),
]

print('EJEMPLOS GUIADOS — Pipeline completo\n' + '='*70)
for consulta, intencion_esperada in consultas_guiadas:
    r = predecir_consulta(consulta)
    acierto = '✓' if intencion_esperada in r.get('intencion', '') else '✗'
    print(f'\n[{acierto}] Consulta:   {consulta}')
    print(f'    Idioma:     {r["idioma"]}')
    print(f'    Intención:  {r["intencion"]}  (esperado: {intencion_esperada})')
    print(f'    Entidades:  {r["entidades"]}')
    print(f'    Limpio:     {r["texto_limpio"]}')

### 4.2 Casos difíciles (fenómenos de la Entrega 1)

In [ ]:
casos_dificiles = [
    ('el pollo frito engorda mucho?',
     'Pragmática: petición implícita de info nutricional o calórica'),
    ('mi bizcocho parece una piedra',
     'Lenguaje figurado: problema de textura → debería ser recipe/cook_time'),
    ('q hago se me kema el arro',
     'Argot + faltas de ortografía: urgencia en la cocina'),
    ('I wanna make chicken al ajillo',
     'Code-switching español/inglés en una misma frase'),
    ('cuantas calorias tiene una papa',
     'Regionalismo: papa → patata (normalización por sinónimo)'),
    ('necesito receta sin gluten y sin lactosa',
     'Términos de dominio que NO deben ser stopwords'),
    ('how many calories does a prawn have',
     'Regionalismo EN: prawn → shrimp (normalización por sinónimo)'),
]

print('CASOS DIFÍCILES — Análisis de comportamiento\n' + '='*70)
for consulta, descripcion in casos_dificiles:
    r = predecir_consulta(consulta)
    print(f'\n[Fenómeno] {descripcion}')
    print(f'  Consulta:  {consulta}')
    print(f'  Idioma:    {r.get("idioma", "—")}')
    print(f'  Intención: {r.get("intencion", "—")}')
    print(f'  Entidades: {r.get("entidades", {})}')
    print(f'  Limpio:    {r.get("texto_limpio", "—")}')

## 5. Propuesta teórica de enrutamiento FAQ (T4)

Al no disponer de una base de datos real de FAQs, el enrutamiento (T4) no puede evaluarse empíricamente. En esta sección se proponen y comparan **dos estrategias teóricas** para cruzar la tupla `[Idioma, Intención, Entidades]` con un hipotético corpus de respuestas.

### Por qué la intención no basta

Dos consultas con la misma intención requieren FAQs completamente distintas:

| Consulta | Intención | Entidades | FAQ necesaria |
|----------|-----------|-----------|---------------|
| ¿Cómo hago pollo al horno? | `recipe` | pollo, hornear | Receta de pollo asado |
| ¿Cómo hago pasta al dente? | `recipe` | pasta, hervir  | Tiempo y técnica de cocción de pasta |
| How long should I boil eggs? | `cook_time` | egg, boil | Tiempos de cocción del huevo |
| How long does pizza take? | `cook_time` | pizza | Tiempo de horneado de pizza |

La **intención** acota el tipo de respuesta; las **entidades** permiten seleccionar la FAQ específica dentro de esa categoría.

### 5.1 Vía léxica: motor de indexación Whoosh

**Whoosh** es un motor de búsqueda de texto completo en Python que crea un índice invertido sobre las preguntas de la base de FAQs y recupera las más relevantes usando el algoritmo **BM25** (variante probabilística de TF-IDF optimizada para recuperación de información).

**Flujo propuesto:**
1. Indexar offline todas las preguntas de las FAQs organizadas por intención.
2. Al recibir una consulta, **filtrar** el índice a las FAQs de la intención detectada en T2 → reduce el espacio de búsqueda.
3. Buscar en ese subconjunto usando los tokens del texto limpio → BM25 rankea por relevancia léxica.
4. Devolver la FAQ con mayor puntuación.

In [ ]:
from whoosh import fields, index, qparser
from whoosh.filedb.filestore import RamStorage
from whoosh.query import And

# Esquema del índice
SCHEMA = fields.Schema(
    faq_id    = fields.ID(stored=True, unique=True),
    intencion = fields.KEYWORD(stored=True),
    pregunta  = fields.TEXT(stored=True),
    respuesta = fields.STORED,
)

# Base de FAQs hipotética (representativa del dominio)
FAQS_DEMO = [
    {'faq_id': '1', 'intencion': 'recipe',
     'pregunta': 'como hacer pollo al horno',
     'respuesta': 'Precalienta el horno a 200°C, sazona el pollo con sal, pimienta y aceite, y hornéalo 45 min.'},
    {'faq_id': '2', 'intencion': 'recipe',
     'pregunta': 'como hacer pasta al dente',
     'respuesta': 'Hierve agua con sal abundante, añade la pasta y cocínala el tiempo indicado en el paquete.'},
    {'faq_id': '3', 'intencion': 'cook_time',
     'pregunta': 'cuanto tiempo tarda el pollo al horno',
     'respuesta': 'El pollo entero tarda unos 20 min por cada 500g a 180°C (pechuga: 25-30 min a 200°C).'},
    {'faq_id': '4', 'intencion': 'cook_time',
     'pregunta': 'cuanto tiempo se hierve el arroz',
     'respuesta': 'El arroz blanco se hierve 18-20 min a fuego medio con tapa y proporción 1:2 (arroz:agua).'},
    {'faq_id': '5', 'intencion': 'ingredient_substitution',
     'pregunta': 'puedo sustituir mantequilla por aceite',
     'respuesta': 'Sí, usa 3/4 de la cantidad en aceite vegetal neutro (girasol o maíz) en lugar de mantequilla.'},
    {'faq_id': '6', 'intencion': 'nutrition_info',
     'pregunta': 'cuantas calorias tiene el pollo',
     'respuesta': 'El pollo cocido sin piel tiene aprox. 165 kcal por 100g; el muslo con piel asciende a 215 kcal.'},
    {'faq_id': '7', 'intencion': 'calories',
     'pregunta': 'cuantas calorias tiene la pasta',
     'respuesta': 'La pasta cocida tiene aprox. 131 kcal por 100g; la pasta seca, unos 350 kcal por 100g.'},
    {'faq_id': '8', 'intencion': 'food_last',
     'pregunta': 'cuanto tiempo dura el queso en la nevera',
     'respuesta': 'Los quesos curados duran 3-4 semanas en nevera; los frescos, 5-7 días tras su apertura.'},
]

# Crear e indexar en memoria
storage = RamStorage()
ix = storage.create_index(SCHEMA)
writer = ix.writer()
for faq in FAQS_DEMO:
    writer.add_document(**faq)
writer.commit()
print(f'Índice Whoosh creado con {len(FAQS_DEMO)} FAQs.')

def buscar_faq_whoosh(query_tuple: dict, top_n: int = 2) -> list:
    intent_name = query_tuple.get('intencion', '').split('(')[-1].rstrip(')')
    texto       = query_tuple.get('texto_limpio', '')

    with ix.searcher() as searcher:
        q_intent = qparser.QueryParser('intencion', ix.schema).parse(intent_name)
        q_texto  = qparser.MultifieldParser(['pregunta'], ix.schema).parse(texto)
        q_final  = And([q_intent, q_texto])
        resultados = searcher.search(q_final, limit=top_n)
        return [
            {'faq_id': r['faq_id'], 'pregunta': r['pregunta'],
             'respuesta': r['respuesta'], 'score_bm25': round(r.score, 3)}
            for r in resultados
        ]

# Demostración
print('\nDEMOSTRACIÓN — Enrutamiento Whoosh\n' + '='*60)
for consulta in ['como hacer pollo al horno', 'cuanto tiempo se tarda en cocer arroz',
                 'cuantas calorias tiene la pasta al dente']:
    q = predecir_consulta(consulta)
    faqs = buscar_faq_whoosh(q)
    print(f'\nConsulta: {consulta}')
    print(f'  → [{q["idioma"]}] Intención: {q["intencion"]} | Entidades: {q["entidades"]}')
    if faqs:
        for f in faqs:
            print(f'  FAQ [{f["faq_id"]}] (BM25={f["score_bm25"]}) {f["pregunta"]}')
            print(f'    Respuesta: {f["respuesta"][:90]}...')
    else:
        print('  Sin resultados (la FAQ no está en el corpus demo).')

### 5.2 Vía semántica: embeddings + similitud coseno

La vía léxica puede fallar ante **paráfrasis** o **distancia léxica**: si el usuario pregunta *"¿cuánto tarda en cocerse la pasta?"* y la FAQ dice *"¿cuánto tiempo debe hervir el espagueti?"*, Whoosh no reconocerá la similitud porque comparten pocos tokens.

La **vía semántica** codifica tanto la consulta del usuario como las preguntas de las FAQs como vectores densos en un espacio semántico compartido (*embeddings*). La FAQ más cercana en ese espacio (mayor **similitud coseno**) es la más relevante, independientemente del vocabulario exacto empleado.

Se propone el modelo **`paraphrase-multilingual-MiniLM-L12-v2`** (SentenceTransformers) por su soporte nativo de español e inglés con un tamaño de modelo manejable (≈120 MB).

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print('Cargando modelo de embeddings...')
modelo_emb = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Pre-calcular embeddings de las FAQs (en producción se haría offline y se almacenarían)
faqs_preguntas   = [f['pregunta']  for f in FAQS_DEMO]
faqs_respuestas  = [f['respuesta'] for f in FAQS_DEMO]
faqs_intenciones = [f['intencion'] for f in FAQS_DEMO]
embeddings_faqs  = modelo_emb.encode(faqs_preguntas, convert_to_numpy=True)
print(f'Embeddings generados: {embeddings_faqs.shape}')

def buscar_faq_semantico(query_tuple: dict, top_n: int = 2) -> list:
    intent_name   = query_tuple.get('intencion', '').split('(')[-1].rstrip(')')
    texto_usuario = query_tuple.get('texto_original', '')

    emb_query = modelo_emb.encode([texto_usuario], convert_to_numpy=True)
    sims      = cosine_similarity(emb_query, embeddings_faqs)[0]

    candidatos = [
        {'faq_id': FAQS_DEMO[i]['faq_id'], 'pregunta': faqs_preguntas[i],
         'respuesta': faqs_respuestas[i], 'coseno': round(float(sims[i]), 3)}
        for i, intencion in enumerate(faqs_intenciones)
        if intencion == intent_name
    ]
    candidatos.sort(key=lambda x: x['coseno'], reverse=True)
    return candidatos[:top_n]

# Demostración con paráfrasis
print('\nDEMOSTRACIÓN — Enrutamiento semántico (paráfrasis)\n' + '='*60)
parafraseos = [
    ('¿cuánto tarda en cocerse la pasta al punto?',  'Paráfrasis de FAQ 4 (arroz/pasta)'),
    ('tiempo de cocción del ave en el horno',         'Paráfrasis de FAQ 3 (pollo/ave)'),
    ('valor calórico de los espaguetis',              'Paráfrasis de FAQ 7 (pasta/espaguetis)'),
]
for consulta, etiqueta in parafraseos:
    q = predecir_consulta(consulta)
    faqs = buscar_faq_semantico(q)
    print(f'\n[{etiqueta}]')
    print(f'  Consulta:  {consulta}')
    print(f'  Intención: {q["intencion"]}')
    if faqs:
        for f in faqs:
            print(f'  FAQ [{f["faq_id"]}] (cos={f["coseno"]}) {f["pregunta"]}')
            print(f'    Respuesta: {f["respuesta"][:90]}...')
    else:
        print('  Sin candidatos con la intención detectada.')

### 5.3 Comparación y recomendación

| Criterio | Whoosh (léxico / BM25) | Embeddings (semántico) |
|----------|------------------------|------------------------|
| **Manejo de paráfrasis** | Bajo — requiere tokens comunes | Alto — captura semántica latente |
| **Velocidad de inferencia** | Muy alta (<10 ms) | Media (50–200 ms con modelo ligero en CPU) |
| **Coste computacional** | Mínimo | Moderado (GPU opcional pero recomendable) |
| **Coste de infraestructura** | Muy bajo (proceso local) | Bajo-medio (almacenar embeddings precomputados) |
| **Interpretabilidad** | Alta — puntuación BM25 explicable | Baja — espacio vectorial denso |
| **Robustez ante variaciones lingüísticas** | Baja | Alta |
| **Ajuste al dominio** | Requiere sinónimos manuales (ya aplicados en T1) | Parcial — modelo multilingüe genérico |

**Recomendación: arquitectura híbrida en dos fases**

1. **Filtro por intención** (T2): reduce el espacio de búsqueda de toda la base de FAQs al subconjunto de la intención detectada. Con 8 intenciones balanceadas, esto reduce el corpus en un factor de ~8×.
2. **Ranking semántico** sobre los candidatos filtrados: los embeddings del modelo multilingüe reordenan los candidatos por similitud coseno. Al operar sobre un corpus pequeño (FAQs de una sola intención), la latencia es asumible.
3. **Desempate léxico con Whoosh** en caso de empate o similitudes muy próximas: proporciona un criterio objetivo basado en la presencia de términos clave.

Esta estrategia mantiene la latencia por debajo del umbral de 200-300 ms establecido en las restricciones no técnicas (Entrega 1, Sección 4) y aprovecha la robustez semántica de los embeddings donde más importa: en el ranking final de candidatos dentro de una misma categoría de intención.